In [ ]:
#Load Data
import pandas as pd
import json

# NewsAPI
with open("dhl_news.json", "r", encoding="utf-8") as f:
    news_data = json.load(f)

news_df = pd.DataFrame(news_data)

# DDGS
ddgs_df = pd.read_csv("ddgs_dhl.csv")

# Supply Chain Dive
supply_df = pd.read_csv("supplychaindive_articles.csv")

print("NewsAPI:", len(news_df))
print("DDGS:", len(ddgs_df))
print("Supply Chain:", len(supply_df))

NewsAPI: 97
DDGS: 194
Supply Chain: 41


In [2]:
import os

print(os.getcwd())
print(os.listdir())

c:\Users\Hima Bindu N\Downloads\NLP
['ddgsdhl.ipynb', 'dhl_dataset.csv', 'dhl_news.json', 'Googlenews.ipynb', 'merge_data.ipynb', 'Newsapi.ipynb', 'Supplychain.ipynb', 'supplychaindive_articles.csv']


In [8]:
#Check Columns
print(news_df.columns)
print(ddgs_df.columns)
print(supply_df.columns)

Index(['title', 'source', 'published_at', 'url', 'description'], dtype='str')
Index(['title', 'href', 'body'], dtype='str')
Index(['title', 'source'], dtype='str')


In [9]:
#Standardize Columns
#NewsAPI
news_df["content"] = news_df["description"]
news_df["source_name"] = "NewsAPI"

# DDGS
ddgs_df["content"] = ddgs_df["body"]
ddgs_df["url"] = ddgs_df["href"]
ddgs_df["source_name"] = "DDGS"

# Supply Chain
supply_df["content"] = supply_df["title"]
supply_df["url"] = ""
supply_df["source_name"] = "SupplyChainDive"

In [10]:
#Select Common Columns
news_df = news_df[["title", "content", "url", "source_name"]]

ddgs_df = ddgs_df[["title", "content", "url", "source_name"]]

supply_df = supply_df[["title", "content", "url", "source_name"]]

In [11]:
#Merge
combined_df = pd.concat(
    [news_df, ddgs_df, supply_df],
    ignore_index=True
)

print("Before duplicates:", len(combined_df))

Before duplicates: 332


In [12]:
#Remove Duplicates
combined_df.drop_duplicates(
    subset=["title"],
    inplace=True
)

print("After duplicates:", len(combined_df))

After duplicates: 314


In [13]:
#Handle Missing Values
combined_df.fillna("", inplace=True)

print(combined_df.isnull().sum())

title          0
content        0
url            0
source_name    0
dtype: int64


In [14]:
#Save Merged Dataset
combined_df.to_csv(
    "merged_dhl_dataset.csv",
    index=False
)

print("Merged dataset saved successfully!")

Merged dataset saved successfully!


In [ ]:
#Verfiy
print(combined_df.head())
print(combined_df.shape)

                                               title  \
0  Hormuz halt forces Middle East trade into huge...   
1  The World’s Best Tequila—According To The 2026...   
2  Property Council 2026 awards: Tauranga’s new M...   
3  The Hidden Infrastructure Challenge Behind Ren...   
4  DHL Says Clean Energy Technology Is Getting Ha...   

                                             content  \
0  A year after a US-Israel and Iran conflict, Si...   
1  This stunner stands out for its exceptional si...   
2  The eight-storey building beat 133 entries in ...   
3  The Middle East crisis has spurred more intere...   
4  Global efforts to transition away from fossil ...   

                                                 url source_name  
0  https://economictimes.indiatimes.com/industry/...     NewsAPI  
1  https://www.forbes.com/sites/bradjaphe/2026/06...     NewsAPI  
2  https://www.nzherald.co.nz/property/property-c...     NewsAPI  
3  https://oilprice.com/Latest-Energy-News/World-...     N

In [16]:
#Create a clean text field
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

combined_df["clean_text"] = (
    combined_df["title"] + " " +
    combined_df["content"]
)

combined_df["clean_text"] = combined_df["clean_text"].apply(clean_text)

In [19]:
#Remove Stopwords
import nltk
nltk.download("stopwords")

from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    return " ".join(
        [w for w in text.split() if w not in stop_words]
    )

combined_df["clean_text"] = combined_df["clean_text"].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to C:\Users\Hima Bindu
[nltk_data]     N\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [20]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    return " ".join(
        [w for w in text.split() if w not in stop_words]
    )

combined_df["clean_text"] = (
    combined_df["clean_text"]
    .apply(remove_stopwords)
)

print(combined_df["clean_text"].head())

0    hormuz halt forces middle east trade huge rewi...
1    world best tequila according 2026 world tequil...
2    property council 2026 awards tauranga new mare...
3    hidden infrastructure challenge behind renewab...
4    dhl says clean energy technology getting harde...
Name: clean_text, dtype: str


In [21]:
combined_df.to_csv(
    "processed_dhl_dataset.csv",
    index=False
)

print("Dataset saved successfully")
print(combined_df.shape)

Dataset saved successfully
(314, 5)


In [22]:
combined_df.shape

(314, 5)

In [1]:
import pandas as pd
import json

# NewsAPI
with open("dhl_news.json", "r", encoding="utf-8") as f:
    news_data = json.load(f)

news_df = pd.DataFrame(news_data)

# DDGS
ddgs_df = pd.read_csv("ddgs_dhl.csv")

# Supply Chain Dive
supply_df = pd.read_csv("supplychaindive_articles.csv")

# Standardize columns
news_df["content"] = news_df["description"]
news_df["source_name"] = "NewsAPI"

ddgs_df["content"] = ddgs_df["body"]
ddgs_df["url"] = ddgs_df["href"]
ddgs_df["source_name"] = "DDGS"

supply_df["content"] = supply_df["title"]
supply_df["url"] = ""
supply_df["source_name"] = "SupplyChainDive"

news_df = news_df[["title","content","url","source_name"]]
ddgs_df = ddgs_df[["title","content","url","source_name"]]
supply_df = supply_df[["title","content","url","source_name"]]

combined_df = pd.concat(
    [news_df, ddgs_df, supply_df],
    ignore_index=True
)

combined_df.drop_duplicates(
    subset=["title"],
    inplace=True
)

print(combined_df.shape)

combined_df.to_csv(
    "merged_dhl_dataset.csv",
    index=False
)

print("Merged dataset saved")

(314, 4)
Merged dataset saved
